<a href="https://colab.research.google.com/github/ashwinsubramaniam230/RL-controller-relative-motion-control-codes/blob/main/RL_CONTROLLER_FYP_ASHWIN_SUBRAMANIAM_KULANTHAYAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install stable-baselines3

In [ ]:
import gymnasium as gym
#import gym
from gym import Env
from gymnasium import spaces
from gym.spaces import Discrete, Box, Dict, Tuple, MultiBinary, MultiDiscrete
import numpy as np
import random
import os
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.evaluation import evaluate_policy

In [ ]:
import math

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.linalg import solve_continuous_are
import matplotlib.pyplot as plt

In [ ]:
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.results_plotter import load_results
import os

In [ ]:
"""
rendezvous_env.py  (2D + atmospheric drag)
==========================================
"""

import math
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from scipy.integrate import solve_ivp


class RendezvousEnv(gym.Env):

    # ── Atmospheric-index ranges ──────────────────────────────────────────
    F107_LOW,  F107_HIGH = 65.0,  250.0
    AP_LOW,    AP_HIGH   =  0.0,   30.0
    H_KM                 = 400.0

    def __init__(self):
        super().__init__()

        # ── Orbital mechanics ─────────────────────────────────────────────
        mu     = 3.986e5 * 1e9
        Re     = 6_371e3
        a      = Re + self.H_KM * 1e3
        self.n = np.sqrt(mu / a**3)
        self.V_t = math.sqrt(mu / a)          # orbital speed [m/s]

        T_orbit       = 2.0 * np.pi / self.n
        self.dt       = 0.05 * T_orbit
        self.max_time = 15.0 * T_orbit

        # ── Spacecraft ────────────────────────────────────────────────────
        self.m0    = 30.0
        self.T_max = 2.5e-3
        self.Isp   = 3300.0
        self.g0    = 9.80665

        # ── Aerodynamics ──────────────────────────────────────────────────
        self.C_D   = 2.2
        self.A_ref = 0.06    # m²
        self._CdA  = self.C_D * self.A_ref

        # ── Convergence ───────────────────────────────────────────────────
        self.c_r = 1.0
        self.c_v = 0.01
        self.C_conv = np.diag([
            1/self.c_r**2, 1/self.c_r**2,
            1/self.c_v**2, 1/self.c_v**2,
        ])

        # ── Initial conditions ────────────────────────────────────────────
        self.pos0_nom = np.array([500.0, -500.0])
        self.vel0_nom = np.array([  1.0,   -1.0])
        self.d0 = np.linalg.norm(self.pos0_nom)   # ≈ 707 m
        self.v0 = np.linalg.norm(self.vel0_nom)   # ≈ 1.41 m/s

        # ── CWH matrices (2D in-plane) ────────────────────────────────────
        n = self.n
        self.A = np.array([
            [0,      0,  1,    0],
            [0,      0,  0,    1],
            [3*n**2, 0,  0,  2*n],
            [0,      0, -2*n,  0],
        ])
        self.B = np.zeros((4, 2))
        self.B[2, 0] = 1.0
        self.B[3, 1] = 1.0

        # ── Spaces  ────────────────────────────────────────────
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(3,), dtype=np.float32
        )
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(4,), dtype=np.float32
        )

        # ── Reward hyperparameters ────────────────────────────
        self._alpha      = 5.0
        self._beta       = 100.0
        self._lambda_dv  = 0.08
        self._R_conv     = 100
        self._R_fail     = 100

    # ─────────────────────────────────────────────────────────────────────
    # Atmospheric density
    # ─────────────────────────────────────────────────────────────────────
    def _compute_density(self, F107, Ap):
        T = 900.0 + 2.5 * (F107 - 70.0) + 1.5 * Ap
        m = 27.0  - 0.012 * (self.H_KM - 200.0)
        H = T / m
        return 6e-10 * math.exp(-(self.H_KM - 175.0) / H)

    # ─────────────────────────────────────────────────────────────────────
    # Drag deceleration  [m/s²]
    # ─────────────────────────────────────────────────────────────────────
    def _compute_drag_accel(self, state, mass):
        """
        Returns a 2-element array [a_drag_R, a_drag_T].

        The chaser's absolute velocity components in RTN are:
          v_R = state[2]               (radial — same as relative, since
                                        target is in circular orbit → v_R_target = 0)
          v_T = V_t + state[3]         (tangential — add orbital speed)

        Drag opposes the absolute velocity vector.
        """
        v_R = state[2]
        v_T = self.V_t + state[3]
        V_chaser = math.sqrt(v_R**2 + v_T**2)
        if V_chaser < 1e-6:
            return np.zeros(2)
        coeff = 0.5 * self.rho * (self._CdA / mass) * V_chaser
        return np.array([-coeff * v_R, -coeff * v_T])

    # ─────────────────────────────────────────────────────────────────────
    # Helpers
    # ─────────────────────────────────────────────────────────────────────
    def _get_obs(self):
        pos = self.state[:2]
        vel = self.state[2:]
        return np.concatenate([pos / self.d0, vel / self.v0]).astype(np.float32)

    def _val(self, state):
        return float(state @ self.C_conv @ state)

    def _potential(self, val):
        return -self._alpha * math.log(1.0 + val / self._beta)

    # ─────────────────────────────────────────────────────────────────────
    # reset()
    # ─────────────────────────────────────────────────────────────────────
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # Randomise atmosphere each episode (domain randomisation)
        self.F107 = float(self.np_random.uniform(self.F107_LOW, self.F107_HIGH))
        self.Ap   = float(self.np_random.uniform(self.AP_LOW,   self.AP_HIGH))
        self.rho  = self._compute_density(self.F107, self.Ap)

        self.state     = np.concatenate([self.pos0_nom, self.vel0_nom])
        self.mass      = float(self.m0)
        self.total_dv  = 0.0
        self.time      = 0.0
        self.traj      = [self.state.copy()]

        self._prev_potential = self._potential(self._val(self.state))

        info = {'F107': self.F107, 'Ap': self.Ap, 'rho': self.rho}
        return self._get_obs(), info

    # ─────────────────────────────────────────────────────────────────────
    # step()
    # ─────────────────────────────────────────────────────────────────────
    def step(self, action):
        action = np.asarray(action, dtype=np.float64)

        # 1. Decode action ─────────────────────────────────────────────────
        u_raw = action[:2]
        f     = float(np.clip((action[2] + 1.0) / 2.0, 0.0, 1.0))

        n_u    = np.linalg.norm(u_raw)
        u_unit = u_raw / n_u if n_u > 1e-12 else np.zeros(2)

        umax    = self.T_max / self.mass
        u_thrust = f * umax * u_unit       # [ux, uy] in m/s²

        # 2. Atmospheric drag ──────────────────────────────────────────────
        a_drag  = self._compute_drag_accel(self.state, self.mass)  # [m/s²], 2-element
        u_total = u_thrust + a_drag                                 # thrust + disturbance

        # 3. CWH numerical integration RK45 ──────────────────────────────────────────
        sol = solve_ivp(
            fun    = lambda t, x: self.A @ x + self.B @ u_total,
            t_span = [0.0, self.dt],
            y0     = self.state,
            method = 'RK45',
            rtol   = 1e-9,
            atol   = 1e-11,
        )
        self.state  = sol.y[:, -1]
        self.time  += self.dt

        # 4. Mass / Δv update ──────────────────────────────────────────────
        u_mag         = np.linalg.norm(u_thrust)
        dm            = (self.mass * u_mag / (self.Isp * self.g0)) * self.dt
        dm            = min(dm, self.mass - 1e-3)
        dv_step       = (self.Isp * self.g0) * math.log(self.mass / (self.mass - dm))
        self.total_dv += dv_step
        self.mass    -= dm

        # 5. Convergence metric ────────────────────────────────────────────
        val      = self._val(self.state)
        distance = float(np.linalg.norm(self.state[:2]))
        rel_vel  = float(np.linalg.norm(self.state[2:]))

        # 6. Terminal conditions ───────────────────────────────────────────
        converged = (distance <= 1.0) and (rel_vel <= 0.01)
        timeout   = (self.time >= self.max_time)
        done      = converged or timeout

        if converged:
            print(f"[CONVERGED] Δv={self.total_dv:.3f} m/s  "
                  f"steps={int(self.time/self.dt)}")
            print(self.rho)

        # 7. Rewards
        curr_potential = self._potential(val)

        r_shape = curr_potential - self._prev_potential

        r_fuel = -self._lambda_dv * dv_step

        if converged:
            r_terminal = self._R_conv / ((self.total_dv))
        elif timeout:
            r_terminal = -self._R_fail
        else:
            r_terminal = 0.0

        reward = r_shape + r_fuel + r_terminal

        self._prev_potential = curr_potential
        self.traj.append(self.state.copy())

        info = dict(
            distance   = distance,
            rel_vel    = rel_vel,
            total_dv   = self.total_dv,
            mass       = self.mass,
            converged  = converged,
            val_conv   = val,
            r_shape    = r_shape,
            r_fuel     = r_fuel,
            r_terminal = r_terminal,
            drag_accel = float(np.linalg.norm(a_drag)),
            drag_vec   = a_drag.tolist(),
            rho        = self.rho,
            F107       = self.F107,
            Ap         = self.Ap,
        )

        return self._get_obs(), float(reward), done, False, info

In [ ]:
import numpy as np
import torch.nn as nn
import csv
import os
from collections import deque

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import (
    EvalCallback,
    CheckpointCallback,
    BaseCallback,
)


# ── Schedules ────────────────────────────────────────────────────────────────

def linear_schedule(v_start: float, v_end: float = 0.0):
    """Linear decay from v_start → v_end over training (paper body text)."""
    def schedule(progress_remaining: float) -> float:
        return v_end + progress_remaining * (v_start - v_end)
    return schedule



class ConvergenceLogger(BaseCallback):
    """Prints convergence rate and mean Δv every `log_every` episodes."""

    def __init__(self, log_every: int = 100):
        super().__init__()
        self.log_every  = log_every
        self.n_eps      = 0
        self.n_conv     = 0
        self.dv_conv    = []

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            ep = info.get("episode")
            if ep is None:
                continue
            self.n_eps += 1
            if info.get("converged", False):
                self.n_conv += 1
                self.dv_conv.append(info.get("total_dv", 0.0))
            if self.n_eps % self.log_every == 0:
                rate    = 100.0 * self.n_conv / self.n_eps
                mean_dv = np.mean(self.dv_conv) if self.dv_conv else float("nan")
                print(f"  [ep {self.n_eps:>6}  step {self.num_timesteps:>9,}]"
                      f"  conv={rate:5.1f}%  mean_Δv={mean_dv:.3f} m/s")
        return True



class RewardLogger(BaseCallback):


    def __init__(self, log_path: str = "reward_log.csv", window: int = 100):
        super().__init__()
        self.log_path    = log_path
        self.window      = window
        self.n_eps       = 0
        self.reward_buf  = deque(maxlen=window)
        self._file       = None
        self._writer     = None

    def _on_training_start(self) -> None:
        self._file   = open(self.log_path, "w", newline="")
        self._writer = csv.writer(self._file)
        self._writer.writerow([
            "timestep", "episode", "episode_reward",
            "mean_reward_100", "episode_length",
            "converged", "total_dv",
        ])

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            ep = info.get("episode")
            if ep is None:
                continue
            self.n_eps     += 1
            ep_reward       = ep["r"]
            ep_length       = ep["l"]
            converged       = int(info.get("converged", False))
            total_dv        = info.get("total_dv", float("nan"))

            self.reward_buf.append(ep_reward)
            mean_r100 = np.mean(self.reward_buf)

            self._writer.writerow([
                self.num_timesteps,
                self.n_eps,
                f"{ep_reward:.4f}",
                f"{mean_r100:.4f}",
                ep_length,
                converged,
                f"{total_dv:.4f}" if not np.isnan(total_dv) else "",
            ])
            self._file.flush()   # flush each episode so data survives crashes
        return True

    def _on_training_end(self) -> None:
        if self._file:
            self._file.close()
        print(f"\n[RewardLogger] Saved reward log → {self.log_path}")


# ── Main training function ────────────────────────────────────────────────────

def train(
    n_layers:        int = 6,
    total_timesteps: int = 1_500_000,
    seed:            int = 0,
    reward_log_path: str = "reward_log.csv",   # ← new parameter
):
    """
    Parameters
    ----------
    n_layers        : hidden layers — paper tested 4, 6, 8
    total_timesteps : 14.5M for RL-only (Table II)
    seed            : random seed
    reward_log_path : CSV path for the RewardLogger callback
    """

    # ── Training / eval environments ──────────────────────────────────────
    def make_env(rank):
        def _init():
            env = RendezvousEnv()
            env = Monitor(env)
            env.reset(seed=seed + rank)
            return env
        return _init

    eval_env = DummyVecEnv([make_env(9999)])

    # ── Network architecture (Table II) ──────────────────────────────────
    hidden = [64] * n_layers

    policy_kwargs = dict(
        net_arch      = dict(pi=hidden, vf=hidden),
        activation_fn = nn.ReLU,
        log_std_init  = float(np.log(0.1)),
    )

    n_steps = 500

    # ── Load checkpoint ───────────────────────────────────────────────────
    env = RendezvousEnv()
    model = PPO(
        policy        = "MlpPolicy",
        env           = RendezvousEnv(),
        learning_rate = linear_schedule(3e-4, 0.0),   # α₀=3e-4 → 0
        clip_range    = linear_schedule(0.2,  0.0),   # ε₀=0.2  → 0
        n_steps       = n_steps,
        batch_size    = 50,          # Table II
        n_epochs      = 10,          # Table II
        gamma         = 1.0,         # Table II — no discounting
        gae_lambda    = 0.95,        # standard GAE
        ent_coef      = 0.01,         # body text — no entropy bonus
        vf_coef       = 0.5,         # standard
        max_grad_norm = 0.5,         # standard gradient clipping
        policy_kwargs = policy_kwargs,
        seed          = seed,
        verbose       = 1,
    )


    print("=" * 60)
    print(f"  PPO  —  RL-only Rendezvous Controller")
    print(f"  Layers      : {n_layers} × 64 nodes, ReLU")
    print(f"  n_envs      : 1  (DummyVecEnv)")
    print(f"  Buffer/upd  : {n_steps}  ({n_steps}×1)")
    print(f"  Minibatches : {n_steps // 50}  per update")
    print(f"  Total steps : {total_timesteps:,}")
    env0 = RendezvousEnv()
    print(f"  Δt          : {env0.dt:.1f} s  (0.05 rev)")
    print(f"  Max ToF     : {env0.max_time/env0.dt:.0f} steps  (10 rev)")
    print("=" * 60)

    # ── Callbacks ─────────────────────────────────────────────────────────
    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path = "./best_model/",
        log_path             = "./logs/",
        eval_freq            = max(50_000 // 1, 1),
        n_eval_episodes      = 20,
        deterministic        = True,
        verbose              = 0,
    )

    ckpt_cb = CheckpointCallback(
        save_freq   = max(100_000 // 1, 1),
        save_path   = "./checkpoints/",
        name_prefix = f"ppo_rendezvous_L{n_layers}",
        verbose     = 0,
    )

    log_cb  = ConvergenceLogger(log_every=100)

    # ── RewardLogger: the new callback that produces the CSV ──────────────
    reward_logger = RewardLogger(log_path=reward_log_path, window=100)

    # ── Train ──────────────────────────────────────────────────────────────
    model.learn(
        total_timesteps = total_timesteps,
        callback        = [eval_cb, ckpt_cb, log_cb, reward_logger],
    )

    # ── Save ───────────────────────────────────────────────────────────────
    model_name = f"ppo_rendezvous_L{n_layers}_final"
    model.save(model_name)
    print(f"\nSaved  →  {model_name}.zip")

    eval_env.close()
    return model


train(n_layers=6, reward_log_path="reward_log.csv")

In [ ]:
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

file_path = '/content/drive/MyDrive/reward_log.csv'

try:
    df = pd.read_csv(file_path)

    # Create figure and force white background
    fig = plt.figure(figsize=(12, 6), facecolor='white')
    ax = fig.add_subplot(111, facecolor='white')

    # Plot the lines
    ax.plot(df['timestep'], df['episode_reward'], label='Episode Reward', alpha=0.4, color='dodgerblue')
    ax.plot(df['timestep'], df['mean_reward_100'], label='Average Reward (Mean 100)', color='darkorange', linewidth=2)

    # --- X-AXIS FORMATTING ---
    # 1. Set interval ticks every 100,000 timesteps (0.1M)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(100000))

    # 2. Format the numbers to show "0.1M", "0.2M", etc., and strip the default 1e6 notation
    def millions_formatter(x, pos):
        return f"{x / 1_000_000:.1f}M"

    ax.xaxis.set_major_formatter(ticker.FuncFormatter(millions_formatter))
    # --------------------------

    # Style Title and Axis labels
    ax.set_title('Rewards vs Timesteps', fontsize=14, fontweight='bold', color='black')
    ax.set_xlabel('Timesteps', fontsize=12, color='black')
    ax.set_ylabel('Reward', fontsize=12, color='black')

    # Force axis ticks to look crisp and black
    ax.tick_params(colors='black', labelsize=10)

    # Rotate the x-labels slightly if they crowd each other due to the tight 0.1M spacing
    # plt.xticks(rotation=45)

    # Fix the Legend layout and font color
    legend = ax.legend(fontsize=11, loc='best', facecolor='white', edgecolor='gray', framealpha=1)
    for text in legend.get_texts():
        text.set_color('black')

    # Grid alignment
    ax.grid(True, linestyle='--', alpha=0.5, color='gray')

    # Save and display
    plt.tight_layout()
    plt.savefig('/content/rewards_plot.png', dpi=300, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.show()

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO

# ── Load model ────────────────────────────────────────────────────────────────
env   = RendezvousEnv()
model = PPO.load('/content/drive/MyDrive/best_model_checkpoints/best_model_babasave.zip',env)
# ── Run one episode ───────────────────────────────────────────────────────────
obs, _ = env.reset()
done   = False

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

# ── Extract raw physical states from env.traj ─────────────────────────────────
# env.traj stores raw [x, y, z, vx, vy, vz] at every step — NOT normalised
traj = np.array(env.traj)          # shape: (N+1, 6)

x  = traj[:, 0]   # [m]
y  = traj[:, 1]   # [m]
#z  = traj[:, 2]   # [m]
vx = traj[:, 2]   # [m/s]
vy = traj[:, 3]   # [m/s]
#vz = traj[:, 5]   # [m/s]

N      = len(traj)
times  = np.arange(N) * env.dt    # real time [s]

# ── Plot 1: 2D trajectory (x-y plane) ────────────────────────────────────────
plt.figure(figsize=(10, 8))
plt.plot(x, y, color='blue', linewidth=2, label='Trajectory')
plt.scatter(x[0], y[0], color='green', s=100, zorder=5, label=f'Start ({x[0]:.0f}, {y[0]:.0f}) m')
print(x[0],y[0])
plt.scatter(0, 0, color='red', marker='*', s=300, zorder=5, label='Target (0, 0)')
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.xlabel('x (radial) [m]', fontsize=12)
plt.ylabel('y (along-track) [m]', fontsize=12)
plt.title('Docking Trajectory (RL controller)\n', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.axis('equal')
plt.tight_layout()
plt.show()

# ── Plot 2: Position vs Time ──────────────────────────────────────────────────
plt.figure(figsize=(10, 5))
plt.plot(times/60, x,  color='blue',  linewidth=2, label='x [m]')
plt.plot(times/60, y,  color='red',   linewidth=2, label='y [m]')
plt.axhline(0, color='black', linestyle='--', alpha=0.4)
plt.xlabel('Time [min]', fontsize=12)
plt.ylabel('Position [m]', fontsize=12)
plt.title('Position against Time (RL controller)\n', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# ── Plot 3: Velocity vs Time ──────────────────────────────────────────────────
plt.figure(figsize=(10, 5))
plt.plot(times/60, vx, color='blue',  linewidth=2, label='$v_{x}$ ')
plt.plot(times/60, vy, color='red',   linewidth=2, label='$v_{y}$ ')
plt.axhline(0, color='black', linestyle='--', alpha=0.4)
plt.xlabel('Time [min]', fontsize=12)
plt.ylabel('Velocity [m/s]', fontsize=12)
plt.title('Velocity against Time (RL controller)\n', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# ── Plot 4: Distance & relative speed vs Time ─────────────────────────────────
distance  = np.linalg.norm(traj[:, :3], axis=1)
rel_speed = np.linalg.norm(traj[:, 3:], axis=1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 12))

ax1.plot(times/60, distance, color='blue', linewidth=2,label='Distance')
ax1.axhline(env.c_r, color='blue', linestyle='--', label=f'tolerance = {env.c_r} m')
ax1.set_ylabel('Distance  [m]', fontsize=12)
#ax1.set_title('Rendezvous Progress', fontsize=14)
ax1.set_xlabel('Time [min]', fontsize=12)
ax1.legend(fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.6)

ax2.plot(times/60, rel_speed, color='blue', linewidth=2,label=' Velocity magnitude')
ax2.axhline(env.c_v, color='blue', linestyle='--', label=f'tolerance= {env.c_v} m/s')
ax2.set_ylabel('Speed [m/s]', fontsize=12)
ax2.set_xlabel('Time [min]', fontsize=12)
ax2.legend(fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Converged : {info['converged']}")
print(f"Final distance  : {info['distance']:.3f} m   (tol: {env.c_r} m)")
print(f"Final rel. speed: {info['rel_vel']:.5f} m/s  (tol: {env.c_v} m/s)")
print(f"Total Δv        : {info['total_dv']:.4f} m/s")
print(f"Final mass      : {info['mass']:.4f} kg  (started: {env.m0} kg)")
print(f"Time-of-flight  : {env.time/60:.1f} min  =  {env.time/env.dt:.0f} steps")
# ── Corrected Slicing for Final Values ────────────────────────────────────────
# (Using -1 to grab the last entry of the trajectory array)
final_x  = traj[-1, 0]
final_y  = traj[-1, 1]
# final_z = traj[-1, 2] # Uncomment if you need Z

# Corrected indices based on [x, y, z, vx, vy, vz]
final_vx = traj[-1, 2]
final_vy = traj[-1, 3]


# ── Print to Console ──────────────────────────────────────────────────────────
print("\n" + "="*40)
print("       FINAL STATE FOR RL CONTROLLER     ")
print("="*40)
print(f" x = {final_x:11.4f} m,  y = {final_y:11.4f} m")
print(f"v_x = {final_vx:10.5f} m/s")
print(f"v_y = {final_vy:10.5f} m/s")
print("="*40)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def rollout_and_plot(model, env=None, deterministic=True):
    if env is None:
        env = RendezvousEnv()

    obs, _ = env.reset()

    times      = []
    accel_x    = []
    accel_y    = []
    accel_mag  = []
    thrust_x   = []
    thrust_y   = []
    thrust_mag = []

    done = False

    while not done:
        action, _ = model.predict(obs, deterministic=deterministic)
        action = np.asarray(action, dtype=np.float64)

        u_raw  = action[:2]
        f      = float(np.clip((action[2] + 1.0) / 2.0, 0.0, 1.0))
        n_u    = np.linalg.norm(u_raw)
        u_unit = u_raw / n_u if n_u > 1e-12 else np.zeros(2)

        umax  = env.T_max / env.mass
        u_vec = f * umax * u_unit
        F_vec = env.mass * u_vec

        times.append(env.time)
        accel_x.append(u_vec[0])
        accel_y.append(u_vec[1])
        accel_mag.append(np.linalg.norm(u_vec))
        thrust_x.append(F_vec[0])
        thrust_y.append(F_vec[1])
        thrust_mag.append(np.linalg.norm(F_vec))

        obs, _, terminated, truncated, info = env.step(action)
        done = terminated or truncated

    t_min  = np.array(times) / 60.0

    # ── milli units ───────────────────────────────────────────────────────
    ax_m   = np.array(accel_x)    * 1e3   # mm/s²
    ay_m   = np.array(accel_y)    * 1e3   # mm/s²
    amag_m = np.array(accel_mag)  * 1e3   # mm/s²
    fx_m   = np.array(thrust_x)   * 1e3   # mN
    fy_m   = np.array(thrust_y)   * 1e3   # mN
    fmag_m = np.array(thrust_mag) * 1e3   # mN

    amax_m  = (env.T_max / env.m0) * 1e3
    T_max_m = env.T_max * 1e3

    # ── Figure 1: Acceleration components (x, y) ─────────────────────────
    fig1, ax1 = plt.subplots(figsize=(10, 4))
    ax1.set_title("Control acceleration components vs Time", fontsize=13, fontweight='bold')
    ax1.plot(t_min, ax_m, color='tab:blue',   linewidth=1.5, label='$a_x$')
    ax1.plot(t_min, ay_m, color='tab:orange', linewidth=1.5, label='$a_y$')
    ax1.axhline( amax_m, color='k', linewidth=0.8, linestyle=':',
                 label=f'$\\pm a_{{max}}$ = {amax_m:.3f} mm/s²')
    ax1.axhline(-amax_m, color='k', linewidth=0.8, linestyle=':')
    ax1.axhline(0, color='gray', linewidth=0.4, linestyle='--')
    ax1.set_ylabel("Acceleration (mm/s²)")
    ax1.set_xlabel("Time (min)")
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    fig1.tight_layout()
    fig1.savefig("accel_components_vs_time.png", dpi=150, bbox_inches='tight')

    # ── Figure 2: Acceleration magnitude ─────────────────────────────────
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    ax2.set_title("Control acceleration magnitude vs Time", fontsize=13, fontweight='bold')
    ax2.plot(t_min, amag_m, color='tab:red', linewidth=1.5, label='$|a|$')
    ax2.axhline(amax_m, color='k', linewidth=0.8, linestyle=':',
                label=f'$a_{{max}}$ = {amax_m:.3f} mm/s²')
    ax2.set_ylabel("|a| (mm/s²)")
    ax2.set_xlabel("Time (min)")
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    fig2.tight_layout()
    fig2.savefig("accel_magnitude_vs_time.png", dpi=150, bbox_inches='tight')

    # ── Figure 3: Thrust components (x, y) ───────────────────────────────
    fig3, ax3 = plt.subplots(figsize=(10, 4))
    ax3.set_title("Thrust components vs Time", fontsize=13)
    ax3.plot(t_min, fx_m, color='tab:blue',   linewidth=1.5, label='$F_x$')
    ax3.plot(t_min, fy_m, color='tab:red', linewidth=1.5, label='$F_y$')
    # ax3.axhline( T_max_m, color='k', linewidth=0.8, linestyle=':',
    #              label=f'$\\pm T_{{max}}$ = {T_max_m:.1f} mN')
    ax3.axhline(-T_max_m, color='k', linewidth=0.8, linestyle=':')
    ax3.axhline(0, color='gray', linewidth=0.4, linestyle='--')
    ax3.set_ylabel("Thrust [mN]")
    ax3.set_xlabel("Time [min]")
    ax3.axhline(T_max_m, color='k', linewidth=0.8, linestyle=':',
                label='Maximum Positive Bound')
    ax3.axhline(-T_max_m, color='k', linewidth=0.8, linestyle=':',
                label='Maximum Negative Bound')
    ax3.legend(fontsize=9)
    ax3.grid(True, alpha=0.3)
    fig3.tight_layout()
    fig3.savefig("thrust_components_vs_time.png", dpi=150, bbox_inches='tight')

    # ── Figure 4: Thrust magnitude ────────────────────────────────────────
    fig4, ax4 = plt.subplots(figsize=(8, 8))
    ax4.set_title("Thrust magnitude vs Time", fontsize=13)
    ax4.plot(t_min, fmag_m, color='tab:red', linewidth=1.5, label='Thrust magnitude')
    ax4.axhline(T_max_m, color='k', linewidth=0.8, linestyle='--',
                label='Maximum limit')
    ax4.set_ylabel("Thrust [mN]")
    ax4.set_xlabel("Time (min)")
    ax4.legend(fontsize=9)
    ax4.grid(True, alpha=0.3)
    ax4.set_ylim(bottom=-0.05 * T_max_m)

    fig4.tight_layout()
    fig4.savefig("thrust_magnitude_vs_time.png", dpi=150, bbox_inches='tight')

    plt.show()
    print(f"Total Δv: {info['total_dv']:.3f} m/s  |  Converged: {info['converged']}")

    return t_min, ax_m, ay_m, amag_m, fx_m, fy_m, fmag_m



rollout_and_plot(model)

In [ ]:
"""
monte_carlo_rl_variable_density.py
====================================
Monte Carlo Simulation — RL  Controller with Variable Atmospheric Drag.
"""

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from scipy.stats import gaussian_kde

# ── Import your environment ───────────────────────────────────────────────────
# from your_env_file import RendezvousEnv


# ─────────────────────────────────────────────
# 1.  HELPERS
# ─────────────────────────────────────────────
def _policy_fn(model):
    if model is None:
        env_tmp = RendezvousEnv()
        def _random(obs):
            return env_tmp.action_space.sample()
        return _random
    def _model(obs):
        action, _ = model.predict(obs, deterministic=True)
        return action
    return _model


def _force_density(env, F107, Ap):
    """Inject fixed space-weather into an already-reset env."""
    env.F107 = float(F107)
    env.Ap   = float(Ap)
    env.rho  = env._compute_density(F107, Ap)


def pad_array(arr, length, fill=np.nan):
    out = np.full(length, fill)
    out[:len(arr)] = arr
    return out


def print_stats_block(name, arr, unit=""):
    """Print mean / std for a 1-D array."""
    arr = np.asarray(arr)
    if arr.size == 0:
        print(f"  {name:<24}: no data")
        return
    print(f"  {name:<24}: mean={np.mean(arr):.4f}{unit}  "
          f"std={np.std(arr):.4f}{unit}")


# ─────────────────────────────────────────────
# 2.  SINGLE EPISODE RUNNER
# ─────────────────────────────────────────────
def run_episode(env, policy, fixed_density=None):
    obs, info = env.reset()

    if fixed_density is not None:
        _force_density(env, *fixed_density)
        obs = env._get_obs()

    traj       = [env.state.copy()]
    times      = []
    states_log = []
    dv_log     = []
    drag_log   = []

    t       = 0.0
    success = False

    while True:
        action                           = policy(obs)
        obs, reward, done, _, step_info = env.step(action)

        t += env.dt
        times.append(t)
        traj.append(env.state.copy())
        states_log.append(env.state.copy())
        dv_log.append(env.total_dv)
        drag_log.append(step_info['drag_accel'])

        if done:
            success = step_info['converged']
            break

    return dict(
        traj       = np.array(traj),
        states     = np.array(states_log),
        times      = np.array(times),
        dv_cum     = np.array(dv_log),
        drag_mag   = np.array(drag_log),
        total_dv   = env.total_dv,
        dock_time  = t if success else None,
        success    = success,
        X0         = np.concatenate([env.pos0_nom, env.vel0_nom]),
        mass_final = env.mass,
        rho        = env.rho,
        F107       = env.F107,
        Ap         = env.Ap,
    )


# ─────────────────────────────────────────────
# 3.  MAIN MC FUNCTION
# ─────────────────────────────────────────────
def run_mc(model, N_MC=1000, seed=42):
    policy = _policy_fn(model)
    env    = RendezvousEnv()

    rho_nom_val = 3*(10**(-12))     # nominal = no drag

    rho_min_ref = env._compute_density(env.F107_LOW, env.AP_LOW)
    rho_max_ref = env._compute_density(env.F107_HIGH, env.AP_HIGH)

    print(f"Density model at h = {env.H_KM} km:")
    print(f"  F10.7 in [{env.F107_LOW}, {env.F107_HIGH}], "
          f"Ap in [{env.AP_LOW}, {env.AP_HIGH}]")
    print(f"  rho_min = {rho_min_ref:.3e} kg/m³")
    print(f"  rho_nom = {rho_nom_val}  (NO DRAG — nominal baseline)")
    print(f"  rho_max = {rho_max_ref:.3e} kg/m³")
    print(f"  Ratio max/min = {rho_max_ref/rho_min_ref:.1f}x\n")

    # ── Monte Carlo runs ──────────────────────────────────────────────────
    env.reset(seed=seed)
    print(f"Running {N_MC} Monte Carlo episodes ...")

    mc_results = []
    for i in range(N_MC):
        res = run_episode(env, policy, fixed_density=None)
        mc_results.append(res)
        if (i + 1) % 20 == 0:
            n_ok = sum(r['success'] for r in mc_results)
            print(f"  [{i+1:>3}/{N_MC}]  success so far: {n_ok}")

    # ── NOMINAL episode: force rho = 0 after reset ───────────────────────
    print(f"\nRunning nominal episode  (rho = 0, NO DRAG) ...")
    obs, _ = env.reset()
    env.rho = 0.0
    env.F107 = float(F107_NOM)
    env.Ap   = float(AP_NOM)
    obs = env._get_obs()

    nom_traj       = [env.state.copy()]
    nom_times      = []
    nom_states_log = []
    nom_dv_log     = []
    nom_drag_log   = []
    nom_t_val      = 0.0
    nom_success    = False

    while True:
        action                           = policy(obs)
        obs, reward, done, _, step_info = env.step(action)
        nom_t_val += env.dt
        nom_times.append(nom_t_val)
        nom_traj.append(env.state.copy())
        nom_states_log.append(env.state.copy())
        nom_dv_log.append(env.total_dv)
        nom_drag_log.append(step_info['drag_accel'])
        if done:
            nom_success = step_info['converged']
            break

    nom = dict(
        traj      = np.array(nom_traj),
        states    = np.array(nom_states_log),
        times     = np.array(nom_times),
        dv_cum    = np.array(nom_dv_log),
        drag_mag  = np.array(nom_drag_log),
        total_dv  = env.total_dv,
        dock_time = nom_t_val if nom_success else None,
        success   = nom_success,
        rho       = 0.0,
        F107      = float(F107_NOM),
        Ap        = float(AP_NOM),
    )

    if nom['success']:
        print(f"  Nominal: converged\n")
    else:
        print(f"  Nominal: TIMEOUT\n")

    # ─────────────────────────────────────────────
    # 4.  STATISTICS
    # ─────────────────────────────────────────────
    successes    = [r for r in mc_results if r['success']]
    failures     = [r for r in mc_results if not r['success']]
    n_success    = len(successes)
    success_rate = n_success / N_MC * 100

    dv_all     = np.array([r['total_dv'] for r in mc_results])
    dock_times = np.array([r['dock_time'] for r in successes]) / 60.0 if successes else np.array([])
    final_pos  = np.array([np.linalg.norm(r['states'][-1, :2]) for r in mc_results])
    final_vel  = np.array([np.linalg.norm(r['states'][-1, 2:]) for r in mc_results])
    rho_all    = np.array([r['rho']  for r in mc_results])
    F107_all   = np.array([r['F107'] for r in mc_results])
    Ap_all     = np.array([r['Ap']   for r in mc_results])

    final_x_all  = np.array([r['states'][-1, 0] for r in mc_results])
    final_y_all  = np.array([r['states'][-1, 1] for r in mc_results])
    final_vx_all = np.array([r['states'][-1, 2] for r in mc_results])
    final_vy_all = np.array([r['states'][-1, 3] for r in mc_results])
    suc_mask     = np.array([r['success'] for r in mc_results])

    dv_all_mean   = np.mean(dv_all)
    dock_t_mean   = np.mean(dock_times) if len(dock_times) else np.nan

    print(f"\n{'='*60}")
    print(f"  MONTE CARLO RESULTS  ({N_MC} runs, RL + variable drag)")
    print(f"{'='*60}")
    print(f"  Success rate       : {n_success}/{N_MC}  ({success_rate:.1f}%)")
    if len(dock_times):
        print(f"  Docking time [min] : {dock_times.mean():.1f} +/- {dock_times.std():.1f}")
        print(f"    min / max        : {dock_times.min():.1f} / {dock_times.max():.1f}")
    print(f"  dV (all runs) [m/s]: {dv_all.mean():.4f} +/- {dv_all.std():.4f}")
    print(f"  rho range          : {rho_all.min():.3e} -- {rho_all.max():.3e} kg/m³")
    print(f"{'='*60}\n")

    print(f"{'─'*60}")
    print(f"  STATISTICS   (all runs)")
    print(f"{'─'*60}")
    print_stats_block("dV - all runs [m/s]",     dv_all,     " m/s")
    print_stats_block("Docking time [min]",      dock_times, " min")
    print(f"{'─'*60}")
    print("  FINAL STATE COMPONENTS  (all runs)")
    print_stats_block("Final x  [m]",    final_x_all,  " m")
    print_stats_block("Final y  [m]",    final_y_all,  " m")
    print_stats_block("Final vx [m/s]",  final_vx_all, " m/s")
    print_stats_block("Final vy [m/s]",  final_vy_all, " m/s")
    print_stats_block("Final |pos| [m]",   final_pos, " m")
    print_stats_block("Final |vel| [m/s]", final_vel, " m/s")
    print(f"{'─'*60}\n")

    # ─────────────────────────────────────────────
    # PLOT HELPERS
    # ─────────────────────────────────────────────
    C_MC    = '#90caf9'
    C_FAIL  = '#ef9a9a'
    C_TGT   = 'red'
    C_START = 'green'
    C_NOM   = 'black'

    max_len = max(len(r['times']) for r in mc_results)
    t_all   = np.array([pad_array(r['times'], max_len) for r in mc_results])
    t_mean  = np.nanmean(t_all, axis=0)

    nom_st = nom['states']
    nom_t  = nom['times']
    X0_nom = np.concatenate([env.pos0_nom, env.vel0_nom])

    # ═══════════════════════════════════════════════════════
    # PLOT 1 — XY TRAJECTORIES
    # ═══════════════════════════════════════════════════════
    fig1, ax = plt.subplots(figsize=(10, 8))
    fig1.patch.set_facecolor('white'); ax.set_facecolor('white')
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_edgecolor('black'); sp.set_linewidth(1.2)

    for r in mc_results:
        ax.plot(r['traj'][:,0], r['traj'][:,1],
                color=C_MC if r['success'] else C_FAIL, lw=0.8, alpha=0.9)

    ax.plot(nom['traj'][:,0], nom['traj'][:,1],
            color=C_NOM, lw=1.5, zorder=8, label='Nominal (rho=3.0x10^(-12))')

    ax.scatter(X0_nom[0], X0_nom[1], color=C_START, s=120, zorder=6,
               edgecolors='black', linewidths=0.8, label='Start (nominal)')
    ax.scatter(0, 0, color=C_TGT, marker='*', s=500, zorder=10,
               edgecolors='black', linewidths=0.5, label='Target (0, 0)')
    ax.axhline(0, color='black', lw=0.7); ax.axvline(0, color='black', lw=0.7)
    ax.set_xlabel('x (radial) [m]',      fontsize=13)
    ax.set_ylabel('y (along-track) [m]', fontsize=13)
    ax.set_title(
        'Monte carlo docking trajectories for 1000 runs (RL under drag disturbance)\n',
        fontsize=12, pad=12)
    ax.grid(True, linestyle='--', alpha=0.5, color='grey')
    ax.tick_params(labelsize=11, direction='out', length=6, width=1.2)

    legend_lines = [
        Line2D([0],[0], color=C_MC,   lw=1.5, label=f'Converged ({n_success})'),
        Line2D([0],[0], color=C_FAIL, lw=1.5, label=f'Failed ({len(failures)})'),
    ]
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles + legend_lines, fontsize=10,
              loc='lower right', framealpha=1.0, edgecolor='grey')
    plt.axis('equal')
    plt.tight_layout()
    plt.savefig('mc_rl_trajectories.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show(); print("Saved: mc_rl_trajectories.png")

    # ═══════════════════════════════════════════════════════
    # PLOT 2 — dV & DOCKING-TIME HISTOGRAMS
    # ═══════════════════════════════════════════════════════
    fig2, axes2 = plt.subplots(1, 2, figsize=(14, 6))
    fig2.patch.set_facecolor('white')
    fig2.suptitle(f'Monte Carlo Statistics — RL + Variable Drag  (N={N_MC})', fontsize=15)

    N_BINS = 20

    # ── dV histogram (all runs) ──────────────────────────────────────────
    ax = axes2[0]; ax.set_facecolor('white')
    ax.hist(dv_all, bins=N_BINS, color='royalblue', edgecolor='black', alpha=0.75
            )
    ax.axvline(dv_all.mean(), color='red', lw=2, ls='--',
               label=f'Mean = {dv_all.mean():.4f} m/s')
    ax.set_xlabel('Total delta-V [m/s] ', fontsize=13)
    ax.set_ylabel('Frequency', fontsize=13)
    ax.set_title('Total delta-V distribution (RL controller)', fontsize=13)
    ax.legend(fontsize=9, loc='upper right', framealpha=0.95)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.tick_params(labelsize=11)
    for sp in ax.spines.values(): sp.set_edgecolor('black')

    # ── Docking-time histogram ──────────────────────────────────────────
    ax = axes2[1]; ax.set_facecolor('white')
    if len(dock_times):
        ax.hist(dock_times, bins=N_BINS, color='darkorange', edgecolor='black',
                alpha=0.75)
        ax.axvline(dock_times.mean(), color='red', lw=2, ls='--',
                   label=f'Mean = {dock_times.mean():.1f} min')
    else:
        ax.text(0.5, 0.5, 'No converged runs', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)
    ax.set_xlabel('Docking time [min]', fontsize=13)
    ax.set_ylabel('Frequency', fontsize=13)
    ax.set_title('Docking time distribution (RL controller) ', fontsize=13)
    ax.legend(fontsize=9, loc='upper right', framealpha=0.95)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.tick_params(labelsize=11)
    for sp in ax.spines.values(): sp.set_edgecolor('black')

    plt.tight_layout()
    plt.savefig('mc_rl_histograms.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show(); print("Saved: mc_rl_histograms.png")

    # ═══════════════════════════════════════════════════════
    # PLOT 3 — DENSITY ANALYSIS
    # ═══════════════════════════════════════════════════════
    fig3, axes3 = plt.subplots(2, 2, figsize=(15, 11))
    fig3.patch.set_facecolor('white')
    fig3.suptitle(
        f'Monte Carlo — Atmospheric Density Analysis  (N={N_MC}, h={env.H_KM} km)\n'
        f'F10.7 ~ U({env.F107_LOW:.0f},{env.F107_HIGH:.0f}),  '
        f'Ap ~ U({env.AP_LOW:.0f},{env.AP_HIGH:.0f})',
        fontsize=14)



    # 3c: dV vs rho — all dots labelled as "Runs"
    ax = axes3[1, 0]; ax.set_facecolor('white')
    ax.scatter(rho_all * 1e10, dv_all,
               c='#5b9bd5', s=40, alpha=0.80, edgecolors='black',
               linewidths=0.4, label='Runs', zorder=3)
    ax.set_xlabel('Density  ρ × 10⁻¹⁰  [kg/m³]', fontsize=12)
    ax.set_ylabel('Total delta-V  [m/s]', fontsize=12)
    ax.set_title('Total delta-V vs Density (RL controller)', fontsize=12)
    ax.legend(fontsize=9); ax.grid(True, linestyle='--', alpha=0.4)
    ax.tick_params(labelsize=10)
    for sp in ax.spines.values(): sp.set_edgecolor('black')


    plt.tight_layout()
    plt.savefig('mc_rl_density_analysis.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show(); print("Saved: mc_rl_density_analysis.png")









    # ═══════════════════════════════════════════════════════
    # PLOT 8 — FINAL POSITION SCATTER + tolerance circle
    # ═══════════════════════════════════════════════════════
    fig8, ax = plt.subplots(figsize=(9, 9))
    fig8.patch.set_facecolor('white'); ax.set_facecolor('white')

    if (~suc_mask).any():
        ax.scatter(final_x_all[~suc_mask], final_y_all[~suc_mask],
                   c='#e74c3c', s=90, marker='x', linewidths=2.2, zorder=3,
                   label=f'Failed ({(~suc_mask).sum()})')
    if suc_mask.any():
        ax.scatter(final_x_all[suc_mask], final_y_all[suc_mask],
                   c='#2ecc71', s=60, edgecolors='black', linewidths=0.5, zorder=4,
                   label=f'Converged ({suc_mask.sum()})')

    ax.scatter(0, 0, color='red', marker='*', s=300, zorder=7,
               edgecolors='darkred', linewidths=0.5, label='Target (0, 0) m')

    pos_tol = env.c_r
    ax.add_patch(Circle((0, 0), pos_tol, fill=False, edgecolor='red',
                         linewidth=2.5, linestyle='--', zorder=5,
                         label=f'Position tolerance  r = {pos_tol} m'))

    all_px = final_x_all
    all_py = final_y_all
    lim = max(np.abs(all_px).max(), np.abs(all_py).max(), pos_tol*3) * 1.15
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect('equal', 'box')
    ax.set_xlabel(' x  (radial)  [m] ',      fontsize=13, color='black')
    ax.set_ylabel(' y  (along-track)  [m] ', fontsize=13, color='black')
    ax.set_title('Terminal position scatter for 1000 runs (RL controller)', fontsize=12, color='black')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(fontsize=10, loc='best', framealpha=1.0, edgecolor='grey')
    ax.tick_params(labelcolor='black', labelsize=11)
    for sp in ax.spines.values(): sp.set_edgecolor('black')

    plt.tight_layout()
    plt.savefig('mc_rl_final_pos.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show(); print("Saved: mc_rl_final_pos.png")

    # ═══════════════════════════════════════════════════════
    # PLOT 9 — FINAL VELOCITY SCATTER + tolerance circle
    # ═══════════════════════════════════════════════════════
    fig9, axes9 = plt.subplots(1, 2, figsize=(16, 7))
    fig9.patch.set_facecolor('white')
    fig9.suptitle(
        f'Terminal velocity scatter for 1000 runs (RL controller) zoomed in (left) and zoomed out(right)\n'
        f'N={N_MC}  |  Success: {success_rate:.1f}%  |  Vel tolerance: r = {env.c_v} m/s',
        fontsize=13, color='black')

    vel_tol  = env.c_v
    nom_fvx  = nom['states'][-1, 3]
    nom_fvy  = nom['states'][-1, 3]

    for ax, title, use_zoom in zip(axes9,
            ['Terminal velocity scatter for 1000 runs (RL controller)', 'Terminal velocity scatter for 1000 runs (RL controller) '],
            [True, False]):
        ax.set_facecolor('white')

        if (~suc_mask).any():
            ax.scatter(final_vx_all[~suc_mask], final_vy_all[~suc_mask],
                       c='#e74c3c', s=120, marker='x', linewidths=2.2, zorder=3,
                       label=f'Failed ({(~suc_mask).sum()})')
        if suc_mask.any():
            ax.scatter(final_vx_all[suc_mask], final_vy_all[suc_mask],
                       c='#2ecc71', s=60, edgecolors='black', linewidths=0.5, zorder=4,
                       label=f'Converged ({suc_mask.sum()})')

        ax.add_patch(Circle((0, 0), vel_tol, fill=False, edgecolor='red',
                             linewidth=2.5, linestyle='--', zorder=5,
                             label=f'Vel tol  r = {vel_tol} m/s'))

        if use_zoom:
            suc_vx = final_vx_all[suc_mask] if suc_mask.any() else np.array([0])
            suc_vy = final_vy_all[suc_mask] if suc_mask.any() else np.array([0])
            spread = max(np.abs(suc_vx).max(), np.abs(suc_vy).max(),
                         abs(nom_fvx), abs(nom_fvy), vel_tol) * 1.6
        else:
            spread = max(np.abs(final_vx_all).max(), np.abs(final_vy_all).max(),
                         vel_tol * 5) * 1.15

        ax.set_xlim(-spread, spread); ax.set_ylim(-spread, spread)
        ax.set_aspect('equal', 'box')
        ax.set_xlabel(r'$v_x$ (radial) [m/s]',      fontsize=12, color='black')
        ax.set_ylabel(r'$v_y$ (along track) [m/s]', fontsize=12, color='black')
        ax.set_title(title, fontsize=12, color='black')
        ax.grid(True, linestyle='--', alpha=0.4)
        ax.legend(fontsize=9, loc='upper right', framealpha=1.0, edgecolor='grey')
        ax.tick_params(labelcolor='black', labelsize=10)
        for sp in ax.spines.values(): sp.set_edgecolor('black')

        if (~suc_mask).any():
            fv_speeds = np.linalg.norm(
                np.column_stack([final_vx_all[~suc_mask], final_vy_all[~suc_mask]]), axis=1)
            if fv_speeds.max() < vel_tol * 0.1:
                ax.annotate(
                    f'All {(~suc_mask).sum()} failed runs\nhave |v|≈0 (fuel exhausted)\n→ stacked at origin',
                    xy=(0, 0), xytext=(vel_tol*1.5, vel_tol*2.5),
                    fontsize=8.5, color='#c0392b',
                    arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.4),
                    bbox=dict(boxstyle='round,pad=0.3', fc='#fdecea', ec='#e74c3c', alpha=0.9))

    plt.tight_layout()
    plt.savefig('mc_rl_final_vel.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show(); print("Saved: mc_rl_final_vel.png")

    # ─────────────────────────────────────────────
    # FINAL SUMMARY
    # ─────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  MONTE CARLO SUMMARY  (N={N_MC}, RL + variable drag)")
    print(f"{'='*60}")
    print(f"  Nominal            : rho ( at rho=3.0x10^(-12))")
    print(f"  Altitude           : {env.H_KM} km")
    print(f"  C_D                : {env.C_D},  A_ref = {env.A_ref} m²")
    print(f"  F10.7 range        : U({env.F107_LOW}, {env.F107_HIGH})")
    print(f"  Ap range           : U({env.AP_LOW}, {env.AP_HIGH})")
    print(f"  rho range sampled  : {rho_all.min():.3e} -- {rho_all.max():.3e} kg/m³")
    print(f"{'─'*60}")
    print(f"  Success rate       : {n_success}/{N_MC}  ({success_rate:.1f}%)")
    if len(dock_times):
        print(f"  Docking time [min] : {dock_times.mean():.1f} +/- {dock_times.std():.1f}")
        print(f"    min / max        : {dock_times.min():.1f} / {dock_times.max():.1f}")
    print(f"{'─'*60}")
    print(f"  dV (all)   [m/s]   : {dv_all.mean():.4f} +/- {dv_all.std():.4f}")
    print(f"    min / max        : {dv_all.min():.4f} / {dv_all.max():.4f}")
    print(f"{'─'*60}")
    print(f"  Final pos (all)    : mean={final_pos.mean():.3f}  std={final_pos.std():.3f} m")
    print(f"  Final vel (all)    : mean={final_vel.mean():.4f}  std={final_vel.std():.4f} m/s")
    print(f"{'─'*60}")
    print(f"  FINAL STATE COMPONENTS  (mean / std, all {N_MC} runs)")
    print(f"    x  [m]   : {final_x_all.mean():.4f} / {final_x_all.std():.4f}")
    print(f"    y  [m]   : {final_y_all.mean():.4f} / {final_y_all.std():.4f}")
    print(f"    vx [m/s] : {final_vx_all.mean():.4f} / {final_vx_all.std():.4f}")
    print(f"    vy [m/s] : {final_vy_all.mean():.4f} / {final_vy_all.std():.4f}")
    print(f"{'='*60}")


    return mc_results, nom


# ─────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────
if __name__ == "__main__":

    mc_results, nom = run_mc(model, N_MC=1000, seed=42)